In [3]:
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import time

BASE_URL = "https://export.arxiv.org/api/query"

params = {
    "search_query": "all:computer",
    "start": 0,
    "max_results": 5
}

# Build query URL safely
query_url = BASE_URL + "?" + urllib.parse.urlencode(params)

request = urllib.request.Request(
    query_url,
    headers={
        "User-Agent": "arxiv-data-project/1.0 (research script)"
    }
)

time.sleep(3)

# Make request
with urllib.request.urlopen(request) as response:
    xml_data = response.read()

# Parse XML
root = ET.fromstring(xml_data)

# arXiv uses Atom namespace
ns = {"atom": "http://www.w3.org/2005/Atom"}

papers = []

for entry in root.findall("atom:entry", ns):
    title = entry.find("atom:title", ns).text.strip()
    summary = entry.find("atom:summary", ns).text.strip()
    published = entry.find("atom:published", ns).text
    
    authors = [
        author.find("atom:name", ns).text
        for author in entry.findall("atom:author", ns)
    ]

    pdf_link = next((link.attrib.get("href") for link in entry.findall("atom:link", ns) 
                if link.attrib.get("title") == "pdf" and link.attrib.get("type") == "application/pdf" and link.attrib.get("rel") == "related"), 
                None
            )

    papers.append({
        "title": title,
        "authors": authors,
        "published": published,
        "summary": summary,
        "pdf_link": pdf_link
    })

# Display parsed results
for paper in papers:
    print("\nTITLE:", paper["title"])
    print("AUTHORS:", ", ".join(paper["authors"]))
    print("PUBLISHED:", paper["published"])
    print("SUMMARY:", paper["summary"][:200], "...")
    print("PDF LINK:", paper["pdf_link"])


TITLE: Vision Based Game Development Using Human Computer Interaction
AUTHORS: S. Sumathi, S. K. Srivatsa, M. Uma Maheswari
PUBLISHED: 2010-02-10T19:46:07Z
SUMMARY: A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long vo ...
PDF LINK: https://arxiv.org/pdf/1002.2191v1

TITLE: From truth to computability I
AUTHORS: Giorgi Japaridze
PUBLISHED: 2004-07-21T03:58:22Z
SUMMARY: The recently initiated approach called computability logic is a formal theory of interactive computation. See a comprehensive online source on the subject at http://www.cis.upenn.edu/~giorgi/cl.html . ...
PDF LINK: https://arxiv.org/pdf/cs/0407054v2

TITLE: Changing Neighbors k Secure Sum Protocol for Secure Multi Party Computation
AUTHORS: Rashid Sheikh, Beerendra Kumar, Durgesh Kumar Mishra
PUBLISHED: 2010-02-11T19:58:10Z
SUMMARY: Secure sum computation of private data inpu

In [4]:
#Converting pdf document to text using PyMuPDF
import fitz
import urllib.request
import os

pdf_dir = "pdfs"
os.makedirs(pdf_dir, exist_ok=True) 

pdf_path = "https://arxiv.org/pdf/1002.2191v1"
local_pdf = os.path.join(pdf_dir, "Vision Based Game Development Using HCI.pdf")
urllib.request.urlretrieve(pdf_path, local_pdf)
    
doc = fitz.open(local_pdf)
text = ""
for page in doc:
    text += page.get_text()

print(text[:500])

(IJCSIS) International Journal of Computer Science and Information Security,  
Vol. 7, No. 1, 2010 
Vision Based Game Development Using Human 
Computer Interaction 
                 Ms.S.Sumathi                                       Dr.S.K.Srivatsa                               Dr.M.Uma Maheswari                           
             Bharath  University                   St.Joseph College of Engineering                   Bharath University 
                Chennai,India                        


In [5]:
# trying pymupdf4llm
import pymupdf4llm
import urllib.request
import os

pdf_dir = "pdfs"
os.makedirs(pdf_dir, exist_ok = True)

pdf_path = "https://arxiv.org/pdf/1002.2191v1"
local_pdf = os.path.join(pdf_dir, "Vision Based Game Development Using HCI.pdf")
urllib.request.urlretrieve(pdf_path, local_pdf)

pages = pymupdf4llm.to_markdown(local_pdf, page_chunks=True)

print(f"Pages extracted: {len(pages)}")
print(pages[0]["metadata"])
print(pages[0]["text"][:500])


OCR disabled because Tesseract language data not found.
Pages extracted: 7
{'format': 'PDF 1.4', 'title': 'Microsoft Word - 07011060 IJCSIS Camera Ready Paper.doc', 'author': 'RAZVI', 'subject': '', 'keywords': '', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationDate': "D:20100202231332+05'00'", 'modDate': "D:20100208131620+05'00'", 'trapped': '', 'encryption': None, 'file_path': 'pdfs\\Vision Based Game Development Using HCI.pdf', 'page_count': 7, 'page_number': 1}
_(IJCSIS) International Journal of Computer Science and Information Security, Vol. 7, No. 1, 2010_ 

## Vision Based Game Development Using Human Computer Interaction 

Bharath  University                   St.Joseph College of Engineering                   Bharath University Chennai,India                                       Chennai,India                                     Chennai,India 

_**Abstract**_ **— A Human Computer Interface (HCI) System for playing games is des

In [6]:
for page in pages:
    if len(page["text"].strip()) <50:
         print(f"Short/empty page: {page['metadata']}")

#checking the character count distribution
lengths = [len(p["text"]) for p in pages]
print(f'Average chars per page: {sum(lengths)/len(lengths):.0f}')
print(f"Min: {min(lengths)}, Max: {max(lengths)}")



Average chars per page: 3487
Min: 2493, Max: 4217


In [7]:
#chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for page in pages:
    splits = text_splitter.split_text(page["text"])

    for s in splits:
        chunks.append({
            "text": s,
            "metadata": page["metadata"]
        })

print(len(chunks))
print(chunks[0])


80
{'text': '_(IJCSIS) International Journal of Computer Science and Information Security, Vol. 7, No. 1, 2010_ \n\n## Vision Based Game Development Using Human Computer Interaction \n\nBharath  University                   St.Joseph College of Engineering                   Bharath University Chennai,India                                       Chennai,India                                     Chennai,India', 'metadata': {'format': 'PDF 1.4', 'title': 'Microsoft Word - 07011060 IJCSIS Camera Ready Paper.doc', 'author': 'RAZVI', 'subject': '', 'keywords': '', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationDate': "D:20100202231332+05'00'", 'modDate': "D:20100208131620+05'00'", 'trapped': '', 'encryption': None, 'file_path': 'pdfs\\Vision Based Game Development Using HCI.pdf', 'page_count': 7, 'page_number': 1}}


In [9]:
import spacy

nlp = spacy.load("en_core_web_sm")

